In [23]:
import pandas as pd
import requests
import os
from datetime import datetime

# Настройки API
indicator_code = "EN.GHG.CO2.PC.CE.AR5"
url = f"https://api.worldbank.org/v2/country/USA/indicator/{indicator_code}?format=json&per_page=100"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    print(f"Запрос к API: {indicator_code}...")
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    raw_data = response.json()

    if isinstance(raw_data, list) and len(raw_data) > 1:
        df_raw = pd.json_normalize(raw_data[1])
        print("Данные успешно загружены в память.")
    else:
        print("API вернул пустой список или ошибку.")
        df_raw = pd.DataFrame()
except Exception as e:
    print(f"Ошибка при запросе: {e}")

Запрос к API: EN.GHG.CO2.PC.CE.AR5...
Данные успешно загружены в память.


In [24]:
if not df_raw.empty:
    col_map = {'date': 'year', 'value': 'value', 'countryiso3code': 'country_iso3'}
    df_clean = df_raw[list(col_map.keys())].copy()
    df_clean.rename(columns=col_map, inplace=True)

    df_clean['indicator'] = indicator_code

    df_clean = df_clean.dropna(subset=['value'])
    df_clean['year'] = df_clean['year'].astype(int)
    df_clean['value'] = pd.to_numeric(df_clean['value'])

    df_clean = df_clean.sort_values('year', ascending=False)

    print(f"Очистка завершена. Готово строк: {len(df_clean)}")
    display(df_clean.head())
else:
    print("Нет данных для очистки.")

Очистка завершена. Готово строк: 55


,year,value,country_iso3,indicator
1,2024,13.619569,USA,EN.GHG.CO2.PC.CE.AR5
2,2023,13.711942,USA,EN.GHG.CO2.PC.CE.AR5
3,2022,14.417715,USA,EN.GHG.CO2.PC.CE.AR5
4,2021,14.316115,USA,EN.GHG.CO2.PC.CE.AR5
5,2020,13.471582,USA,EN.GHG.CO2.PC.CE.AR5


In [25]:
if 'df_clean' in locals():
    general_name = "usa_co2_emissions"
    unique_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"normalized_co2_{unique_suffix}.csv"

    output_dir = os.path.join("..", "data", "normalized", general_name)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(output_dir, filename)

    df_clean.to_csv(output_path, index=False)
    print(f"Создан новый уникальный файл:\n{output_path}")

Создан новый уникальный файл:
..\data\normalized\usa_co2_emissions\normalized_co2_20260319_113427.csv
